In [1]:
import pandas as pd
import numpy as np
import os

# Chemins des dossiers
RAW = "../data/raw/"
PROCESSED = "../data/processed/"

# Créer le dossier processed s'il n'existe pas
os.makedirs(PROCESSED, exist_ok=True)

# Départements bretons
DEPTS_BRETAGNE = ['22', '29', '35', '56']

# Mapping candidats → groupes politiques
MAPPING_2017 = {
    'ARTHAUD'       : 'Ext_gauche',
    'POUTOU'        : 'Ext_gauche',
    'MÉLENCHON'     : 'Gauche',
    'HAMON'         : 'Gauche',
    'JADOT'         : 'Gauche',
    'MACRON'        : 'Centre',
    'FILLON'        : 'Droite',
    'LE PEN'        : 'Ext_droite',
    'DUPONT-AIGNAN' : 'Ext_droite',
    'LASSALLE'      : 'Ext_droite',
    'ASSELINEAU'    : 'Ext_droite',
    'CHEMINADE'     : 'Ext_droite',
}

MAPPING_2022 = {
    'ARTHAUD'       : 'Ext_gauche',
    'POUTOU'        : 'Ext_gauche',
    'MÉLENCHON'     : 'Gauche',
    'HIDALGO'       : 'Gauche',
    'JADOT'         : 'Gauche',
    'ROUSSEL'       : 'Gauche',
    'MACRON'        : 'Centre',
    'PÉCRESSE'      : 'Droite',
    'LE PEN'        : 'Ext_droite',
    'DUPONT-AIGNAN' : 'Ext_droite',
    'ZEMMOUR'       : 'Ext_droite',
    'LASSALLE'      : 'Ext_droite',
}

print("Configuration OK !")
print(f"Dossier raw    : {os.path.abspath(RAW)}")
print(f"Dossier processed : {os.path.abspath(PROCESSED)}")

Configuration OK !
Dossier raw    : d:\MSPR1\projet\electio-analytics-bretagne\data\raw
Dossier processed : d:\MSPR1\projet\electio-analytics-bretagne\data\processed


In [2]:
# ============================================================
# CHARGEMENT FICHIERS ELECTORAUX
# ============================================================
# On charge les resultats de la presidentielle 2017 et 2022
# Ces fichiers contiennent les scores de chaque candidat
# par commune pour toute la France entiere.
# On filtrera ensuite uniquement les communes bretonnes.
#
# pd.read_excel : charge un fichier Excel dans un DataFrame
# header=3 pour 2017 : les 3 premieres lignes sont des titres
#           parasites donc on commence a lire a partir de la
#           ligne 3 pour avoir les vraies colonnes
# header=0 pour 2022 : les colonnes sont bien en premiere ligne
# df.shape : affiche le nombre de lignes et colonnes
# df.columns : affiche les noms des colonnes
# ============================================================

# --- 2017 ---
df_2017 = pd.read_excel(
    RAW + "Presidentielle_2017_Resultats_Communes_Tour_1_c.xls",
    header=3
)
print("2017 charge")
print("Dimensions        :", df_2017.shape)
print("10 premieres colonnes :")
print(df_2017.columns.tolist()[:10])
print()

# --- 2022 ---
df_2022 = pd.read_excel(
    RAW + "resultats-par-niveau-subcom-t1-france-entiere.xlsx",
    header=0
)
print("2022 charge")
print("Dimensions        :", df_2022.shape)
print("10 premieres colonnes :")
print(df_2022.columns.tolist()[:10])

2017 charge
Dimensions        : (35719, 95)
10 premieres colonnes :
['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs']

2022 charge
Dimensions        : (35245, 103)
10 premieres colonnes :
['Code du département', 'Libellé du département', 'Code de la commune', 'Libellé de la commune', 'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins']


In [3]:
# ============================================================
# FILTRAGE DES COMMUNES BRETONNES
# ============================================================
# Les fichiers contiennent toutes les communes de France.
# On garde uniquement les communes des 4 departements
# bretons : 22 (Cotes-d-Armor), 29 (Finistere),
# 35 (Ille-et-Vilaine), 56 (Morbihan).
#
# Le code departement dans le fichier 2017 est un entier
# donc on le convertit en string avec astype(str) pour
# pouvoir le comparer avec notre liste DEPTS_BRETAGNE.
#
# isin() : verifie si la valeur est dans une liste
# .copy() : cree une copie independante du DataFrame
#           pour eviter les avertissements pandas
# ============================================================

# Filtrage 2017
df_2017['Code du département'] = df_2017['Code du département'].astype(str).str.strip()
bretagne_2017 = df_2017[df_2017['Code du département'].isin(DEPTS_BRETAGNE)].copy()
bretagne_2017['annee'] = 2017

print("2017 - toute la France :", len(df_2017), "communes")
print("2017 - Bretagne        :", len(bretagne_2017), "communes")

print()

# Filtrage 2022
df_2022['Code du département'] = df_2022['Code du département'].astype(str).str.strip()
bretagne_2022 = df_2022[df_2022['Code du département'].isin(DEPTS_BRETAGNE)].copy()
bretagne_2022['annee'] = 2022

print("2022 - toute la France :", len(df_2022), "communes")
print("2022 - Bretagne        :", len(bretagne_2022), "communes")

2017 - toute la France : 35719 communes
2017 - Bretagne        : 1233 communes

2022 - toute la France : 35245 communes
2022 - Bretagne        : 1207 communes


In [4]:
# ============================================================
# CALCUL DES SCORES PAR GROUPE POLITIQUE
# ============================================================
# Les fichiers electoraux donnent les scores candidat
# par candidat. On doit regrouper les candidats selon
# nos 5 groupes politiques definis dans le mapping.
#
# Le fichier est en format large : chaque candidat occupe
# plusieurs colonnes (Nom, Prenom, Voix, % Voix/Ins,
# % Voix/Exp) qui se repetent pour chaque candidat.
#
# La strategie est la suivante :
# 1. On repere les colonnes Nom et % Voix/Exp
# 2. Pour chaque candidat on recupere son nom et son score
# 3. On applique le mapping pour trouver son groupe
# 4. On additionne les scores du meme groupe
#
# Cette fonction fait ce travail pour une ligne du fichier
# c est a dire une commune et retourne les 5 scores de groupe
# ============================================================

def calculer_scores(row, mapping):

    # Initialiser les scores a zero pour chaque groupe
    scores = {
        'score_ext_gauche' : 0.0,
        'score_gauche'     : 0.0,
        'score_centre'     : 0.0,
        'score_droite'     : 0.0,
        'score_ext_droite' : 0.0,
    }

    # Recuperer toutes les colonnes de la ligne
    colonnes = row.index.tolist()

    # Parcourir les colonnes pour trouver les colonnes Nom
    for i, col in enumerate(colonnes):
        if col == 'Nom':
            nom_candidat = str(row[col]).strip().upper()

            # Chercher la colonne % Voix/Exp qui suit ce Nom
            for j in range(i, min(i + 6, len(colonnes))):
                if '% Voix/Exp' in str(colonnes[j]):
                    score = row[colonnes[j]]

                    # Verifier que le score est un nombre valide
                    if pd.notna(score) and str(score) != 'nan':
                        try:
                            score = float(str(score).replace(',', '.'))
                            # Chercher le groupe du candidat dans le mapping
                            for cle, groupe in mapping.items():
                                if cle in nom_candidat:
                                    scores['score_' + groupe.lower()] += score
                        except ValueError:
                            pass
                    break

    return pd.Series(scores)


# Test sur les 5 premieres communes bretonnes de 2017
test = bretagne_2017.head(5).apply(
    lambda row: calculer_scores(row, MAPPING_2017), axis=1
)
print("Test calcul scores 2017 :")
print(test)

Test calcul scores 2017 :
      score_ext_gauche  score_gauche  score_centre  score_droite  \
7724               0.0          0.00          0.00          23.8   
7725               0.0          0.00         33.78           0.0   
7726               0.0         25.44          0.00           0.0   
7727               0.0         25.72          0.00           0.0   
7728               0.0         24.84          0.00           0.0   

      score_ext_droite  
7724               0.0  
7725               0.0  
7726               0.0  
7727               0.0  
7728               0.0  


In [5]:
# ============================================================
# DIAGNOSTIC DES COLONNES
# ============================================================
# On affiche toutes les colonnes du fichier 2017 pour
# comprendre la structure exacte et corriger le problème.
# Le fichier Excel a des colonnes dupliquées car chaque
# candidat a ses propres colonnes Nom et % Voix/Exp.
# Pandas renomme automatiquement les doublons en ajoutant
# .1 .2 .3 etc à la fin du nom de colonne.
# On doit donc chercher toutes les variantes de Nom
# et % Voix/Exp dans les colonnes.
# ============================================================

print("Toutes les colonnes du fichier 2017 :")
print()
for i, col in enumerate(bretagne_2017.columns):
    print(f"{i:3d} | {col}")

Toutes les colonnes du fichier 2017 :

  0 | Code du département
  1 | Libellé du département
  2 | Code de la commune
  3 | Libellé de la commune
  4 | Inscrits
  5 | Abstentions
  6 | % Abs/Ins
  7 | Votants
  8 | % Vot/Ins
  9 | Blancs
 10 | % Blancs/Ins
 11 | % Blancs/Vot
 12 | Nuls
 13 | % Nuls/Ins
 14 | % Nuls/Vot
 15 | Exprimés
 16 | % Exp/Ins
 17 | % Exp/Vot
 18 | N°Panneau
 19 | Sexe
 20 | Nom
 21 | Prénom
 22 | Voix
 23 | % Voix/Ins
 24 | % Voix/Exp
 25 | N°Panneau.1
 26 | Sexe.1
 27 | Nom.1
 28 | Prénom.1
 29 | Voix.1
 30 | % Voix/Ins.1
 31 | % Voix/Exp.1
 32 | N°Panneau.2
 33 | Sexe.2
 34 | Nom.2
 35 | Prénom.2
 36 | Voix.2
 37 | % Voix/Ins.2
 38 | % Voix/Exp.2
 39 | N°Panneau.3
 40 | Sexe.3
 41 | Nom.3
 42 | Prénom.3
 43 | Voix.3
 44 | % Voix/Ins.3
 45 | % Voix/Exp.3
 46 | N°Panneau.4
 47 | Sexe.4
 48 | Nom.4
 49 | Prénom.4
 50 | Voix.4
 51 | % Voix/Ins.4
 52 | % Voix/Exp.4
 53 | N°Panneau.5
 54 | Sexe.5
 55 | Nom.5
 56 | Prénom.5
 57 | Voix.5
 58 | % Voix/Ins.5
 59 | % Vo

In [6]:
# ============================================================
# IDENTIFIER LES COLONNES CANDIDATS
# ============================================================
# On cherche toutes les colonnes qui contiennent "Nom"
# et toutes celles qui contiennent "% Voix/Exp"
# pour comprendre exactement comment elles sont nommées.
# Pandas a ajouté .1 .2 .3 etc pour les colonnes dupliquées.
# On va utiliser ces noms exacts pour extraire les scores.
# ============================================================

# Colonnes contenant Nom
cols_nom = [col for col in bretagne_2017.columns if 'Nom' in str(col)]
print("Colonnes Nom :")
print(cols_nom)

print()

# Colonnes contenant % Voix/Exp
cols_score = [col for col in bretagne_2017.columns if '% Voix/Exp' in str(col)]
print("Colonnes % Voix/Exp :")
print(cols_score)

Colonnes Nom :
['Nom', 'Nom.1', 'Nom.2', 'Nom.3', 'Nom.4', 'Nom.5', 'Nom.6', 'Nom.7', 'Nom.8', 'Nom.9', 'Nom.10']

Colonnes % Voix/Exp :
['% Voix/Exp', '% Voix/Exp.1', '% Voix/Exp.2', '% Voix/Exp.3', '% Voix/Exp.4', '% Voix/Exp.5', '% Voix/Exp.6', '% Voix/Exp.7', '% Voix/Exp.8', '% Voix/Exp.9', '% Voix/Exp.10']


In [7]:
# ============================================================
# FONCTION CORRIGEE CALCUL DES SCORES
# ============================================================
# Maintenant qu'on connait la structure exacte du fichier
# on peut corriger la fonction.
#
# Le fichier a 11 candidats maximum par commune.
# Pour chaque candidat il y a une paire de colonnes :
#   - Nom, Nom.1, Nom.2 ... Nom.10
#   - % Voix/Exp, % Voix/Exp.1 ... % Voix/Exp.10
#
# Pour chaque paire on recupere le nom du candidat
# et son score en % des voix exprimees.
# On cherche ensuite ce nom dans le mapping pour
# trouver son groupe politique et on additionne
# les scores du meme groupe.
# ============================================================

def calculer_scores_v2(row, mapping, cols_nom, cols_score):

    # Initialiser les scores a zero pour chaque groupe
    scores = {
        'score_ext_gauche' : 0.0,
        'score_gauche'     : 0.0,
        'score_centre'     : 0.0,
        'score_droite'     : 0.0,
        'score_ext_droite' : 0.0,
    }

    # Parcourir chaque paire Nom / % Voix/Exp
    for col_nom, col_score in zip(cols_nom, cols_score):

        # Recuperer le nom du candidat
        nom = str(row[col_nom]).strip().upper()

        # Recuperer le score du candidat
        score = row[col_score]

        # Verifier que le score est valide
        if pd.isna(score):
            continue

        try:
            score = float(str(score).replace(',', '.'))
        except ValueError:
            continue

        # Chercher le groupe du candidat dans le mapping
        for cle, groupe in mapping.items():
            if cle in nom:
                scores['score_' + groupe.lower()] += score
                break

    return pd.Series(scores)


# Colonnes Nom et % Voix/Exp du fichier 2017
cols_nom_2017   = [col for col in bretagne_2017.columns if 'Nom' in str(col)]
cols_score_2017 = [col for col in bretagne_2017.columns if '% Voix/Exp' in str(col)]

# Test sur les 5 premieres communes bretonnes de 2017
test = bretagne_2017.head(5).apply(
    lambda row: calculer_scores_v2(
        row, MAPPING_2017, cols_nom_2017, cols_score_2017
    ),
    axis=1
)
print("Test calcul scores 2017 :")
print(test)

Test calcul scores 2017 :
      score_ext_gauche  score_gauche  score_centre  score_droite  \
7724              2.84         22.38         22.95         23.80   
7725              2.43         24.63         33.78         18.30   
7726              3.92         32.38         24.73          9.07   
7727              3.45         34.78         24.21         11.42   
7728              3.11         35.62         22.88         14.22   

      score_ext_droite  
7724             28.04  
7725             20.86  
7726             29.89  
7727             26.14  
7728             24.18  


In [8]:
# ============================================================
# APPLICATION SUR TOUTES LES COMMUNES
# ============================================================
# On applique la fonction sur toutes les communes bretonnes
# pour 2017 et 2022.
#
# On recupere aussi les colonnes utiles de chaque fichier :
# - code_commune : identifiant unique de la commune
# - nom_commune : nom de la commune
# - annee : 2017 ou 2022
# - nb_inscrits : nombre d inscrits sur les listes
# - nb_votants : nombre de personnes ayant vote
# - taux_abstention : pourcentage d abstention
#
# pd.concat : fusionne les colonnes de base avec les
# scores calcules pour former un seul DataFrame
# reset_index : remet les index a zero apres le filtrage
# ============================================================

# --- TRAITEMENT 2017 ---

# Colonnes Nom et % Voix/Exp du fichier 2017
cols_nom_2017   = [c for c in bretagne_2017.columns if 'Nom' in str(c)]
cols_score_2017 = [c for c in bretagne_2017.columns if '% Voix/Exp' in str(c)]

# Calcul des scores pour toutes les communes 2017
scores_2017 = bretagne_2017.apply(
    lambda row: calculer_scores_v2(
        row, MAPPING_2017, cols_nom_2017, cols_score_2017
    ),
    axis=1
)

# Construction du DataFrame 2017 avec colonnes utiles
base_2017 = pd.DataFrame({
    'code_commune'   : bretagne_2017['Code de la commune'].astype(str).str.zfill(3),
    'nom_commune'    : bretagne_2017['Libellé de la commune'],
    'code_dept'      : bretagne_2017['Code du département'],
    'annee'          : 2017,
    'nb_inscrits'    : bretagne_2017['Inscrits'],
    'nb_votants'     : bretagne_2017['Votants'],
    'taux_abstention': bretagne_2017['% Abs/Ins'],
}).reset_index(drop=True)

# Fusion colonnes de base + scores
electoral_2017 = pd.concat([base_2017, scores_2017.reset_index(drop=True)], axis=1)

print("2017 traite :", electoral_2017.shape)
print(electoral_2017.head(3))

print()

# --- TRAITEMENT 2022 ---

# Colonnes Nom et % Voix/Exp du fichier 2022
cols_nom_2022   = [c for c in bretagne_2022.columns if 'Nom' in str(c)]
cols_score_2022 = [c for c in bretagne_2022.columns if '% Voix/Exp' in str(c)]

# Calcul des scores pour toutes les communes 2022
scores_2022 = bretagne_2022.apply(
    lambda row: calculer_scores_v2(
        row, MAPPING_2022, cols_nom_2022, cols_score_2022
    ),
    axis=1
)

# Construction du DataFrame 2022 avec colonnes utiles
base_2022 = pd.DataFrame({
    'code_commune'   : bretagne_2022['Code de la commune'].astype(str).str.zfill(3),
    'nom_commune'    : bretagne_2022['Libellé de la commune'],
    'code_dept'      : bretagne_2022['Code du département'],
    'annee'          : 2022,
    'nb_inscrits'    : bretagne_2022['Inscrits'],
    'nb_votants'     : bretagne_2022['Votants'],
    'taux_abstention': bretagne_2022['% Abs/Ins'],
}).reset_index(drop=True)

# Fusion colonnes de base + scores
electoral_2022 = pd.concat([base_2022, scores_2022.reset_index(drop=True)], axis=1)

print("2022 traite :", electoral_2022.shape)
print(electoral_2022.head(3))

2017 traite : (1233, 12)
  code_commune nom_commune code_dept  annee  nb_inscrits  nb_votants  \
0          001    Allineuc        22   2017          419         361   
1          002       Andel        22   2017          858         761   
2          003    Aucaleuc        22   2017          672         580   

   taux_abstention  score_ext_gauche  score_gauche  score_centre  \
0            13.84              2.84         22.38         22.95   
1            11.31              2.43         24.63         33.78   
2            13.69              3.92         32.38         24.73   

   score_droite  score_ext_droite  
0         23.80             28.04  
1         18.30             20.86  
2          9.07             29.89  

2022 traite : (1207, 12)
  code_commune nom_commune code_dept  annee  nb_inscrits  nb_votants  \
0          001    Allineuc        22   2022          455         373   
1          002       Andel        22   2022          930         775   
2          003    Aucaleuc 

In [9]:
# ============================================================
# DIAGNOSTIC NOMS CANDIDATS 2022
# ============================================================
# Les scores 2022 sont a zero ce qui signifie que les noms
# des candidats dans le fichier ne correspondent pas
# exactement aux celles du mapping.
# On affiche les noms reels des candidats dans le fichier
# pour voir exactement comment ils sont ecrits.
# ============================================================

# Colonnes Nom du fichier 2022
cols_nom_2022 = [c for c in bretagne_2022.columns if 'Nom' in str(c)]

# Afficher les noms des candidats sur la premiere commune
print("Noms des candidats dans le fichier 2022 :")
premiere_commune = bretagne_2022.iloc[0]
for col in cols_nom_2022:
    print(f"  {col} : {premiere_commune[col]}")

Noms des candidats dans le fichier 2022 :
  Nom : ARTHAUD


In [10]:
# ============================================================
# DIAGNOSTIC COMPLET COLONNES 2022
# ============================================================
# On affiche toutes les colonnes du fichier 2022 pour
# voir exactement comment sont nommees les colonnes
# des candidats. Le fichier 2022 a peut etre une
# structure differente du fichier 2017.
# ============================================================

print("Toutes les colonnes du fichier 2022 :")
print()
for i, col in enumerate(bretagne_2022.columns):
    print(f"{i:3d} | {col}")

Toutes les colonnes du fichier 2022 :

  0 | Code du département
  1 | Libellé du département
  2 | Code de la commune
  3 | Libellé de la commune
  4 | Etat saisie
  5 | Inscrits
  6 | Abstentions
  7 | % Abs/Ins
  8 | Votants
  9 | % Vot/Ins
 10 | Blancs
 11 | % Blancs/Ins
 12 | % Blancs/Vot
 13 | Nuls
 14 | % Nuls/Ins
 15 | % Nuls/Vot
 16 | Exprimés
 17 | % Exp/Ins
 18 | % Exp/Vot
 19 | N°Panneau
 20 | Sexe
 21 | Nom
 22 | Prénom
 23 | Voix
 24 | % Voix/Ins
 25 | % Voix/Exp
 26 | Unnamed: 26
 27 | Unnamed: 27
 28 | Unnamed: 28
 29 | Unnamed: 29
 30 | Unnamed: 30
 31 | Unnamed: 31
 32 | Unnamed: 32
 33 | Unnamed: 33
 34 | Unnamed: 34
 35 | Unnamed: 35
 36 | Unnamed: 36
 37 | Unnamed: 37
 38 | Unnamed: 38
 39 | Unnamed: 39
 40 | Unnamed: 40
 41 | Unnamed: 41
 42 | Unnamed: 42
 43 | Unnamed: 43
 44 | Unnamed: 44
 45 | Unnamed: 45
 46 | Unnamed: 46
 47 | Unnamed: 47
 48 | Unnamed: 48
 49 | Unnamed: 49
 50 | Unnamed: 50
 51 | Unnamed: 51
 52 | Unnamed: 52
 53 | Unnamed: 53
 54 | Unnamed:

In [11]:
# ============================================================
# AFFICHER TOUTES LES COLONNES 2022
# ============================================================
# Le fichier 2022 semble avoir des colonnes Unnamed
# ce qui signifie que les noms des colonnes candidats
# ne se repetent pas comme dans le fichier 2017.
# On affiche toutes les colonnes pour voir la structure
# exacte entre la colonne 19 et la fin du fichier.
# ============================================================

print("Colonnes 19 a 103 du fichier 2022 :")
print()
for i, col in enumerate(bretagne_2022.columns):
    if i >= 19:
        print(f"{i:3d} | {col}")
        

Colonnes 19 a 103 du fichier 2022 :

 19 | N°Panneau
 20 | Sexe
 21 | Nom
 22 | Prénom
 23 | Voix
 24 | % Voix/Ins
 25 | % Voix/Exp
 26 | Unnamed: 26
 27 | Unnamed: 27
 28 | Unnamed: 28
 29 | Unnamed: 29
 30 | Unnamed: 30
 31 | Unnamed: 31
 32 | Unnamed: 32
 33 | Unnamed: 33
 34 | Unnamed: 34
 35 | Unnamed: 35
 36 | Unnamed: 36
 37 | Unnamed: 37
 38 | Unnamed: 38
 39 | Unnamed: 39
 40 | Unnamed: 40
 41 | Unnamed: 41
 42 | Unnamed: 42
 43 | Unnamed: 43
 44 | Unnamed: 44
 45 | Unnamed: 45
 46 | Unnamed: 46
 47 | Unnamed: 47
 48 | Unnamed: 48
 49 | Unnamed: 49
 50 | Unnamed: 50
 51 | Unnamed: 51
 52 | Unnamed: 52
 53 | Unnamed: 53
 54 | Unnamed: 54
 55 | Unnamed: 55
 56 | Unnamed: 56
 57 | Unnamed: 57
 58 | Unnamed: 58
 59 | Unnamed: 59
 60 | Unnamed: 60
 61 | Unnamed: 61
 62 | Unnamed: 62
 63 | Unnamed: 63
 64 | Unnamed: 64
 65 | Unnamed: 65
 66 | Unnamed: 66
 67 | Unnamed: 67
 68 | Unnamed: 68
 69 | Unnamed: 69
 70 | Unnamed: 70
 71 | Unnamed: 71
 72 | Unnamed: 72
 73 | Unnamed: 73
 74 

In [12]:
# ============================================================
# DIAGNOSTIC EN-TETE FICHIER 2022
# ============================================================
# Le fichier 2022 a des colonnes Unnamed car Excel a
# fusionne les cellules d en-tete pour chaque candidat.
# On recharge le fichier sans traitement d en-tete pour
# voir les vraies lignes du fichier et comprendre
# a quelle ligne se trouvent les vrais noms de colonnes.
# ============================================================

# Recharger sans traitement d en-tete pour voir la structure
df_2022_raw = pd.read_excel(
    RAW + "resultats-par-niveau-subcom-t1-france-entiere.xlsx",
    header=None
)

print("5 premieres lignes brutes du fichier 2022 :")
print()
# Afficher les colonnes 19 a 35 pour voir la zone candidats
print(df_2022_raw.iloc[:5, 19:36])

5 premieres lignes brutes du fichier 2022 :

          19    20       21        22    23          24          25   26   27  \
0  N°Panneau  Sexe      Nom    Prénom  Voix  % Voix/Ins  % Voix/Exp  NaN  NaN   
1          1     F  ARTHAUD  Nathalie     3        0.47        0.58  2.0    M   
2          1     F  ARTHAUD  Nathalie     2        0.94        1.17  2.0    M   
3          1     F  ARTHAUD  Nathalie    38        0.43        0.58  2.0    M   
4          1     F  ARTHAUD  Nathalie     8        0.62        0.78  2.0    M   

        28      29     30    31    32   33   34      35  
0      NaN     NaN    NaN   NaN   NaN  NaN  NaN     NaN  
1  ROUSSEL  Fabien    6.0  0.93  1.15  3.0    M  MACRON  
2  ROUSSEL  Fabien    7.0  3.29  4.09  3.0    M  MACRON  
3  ROUSSEL  Fabien  173.0  1.97  2.64  3.0    M  MACRON  
4  ROUSSEL  Fabien   19.0  1.48  1.85  3.0    M  MACRON  


In [13]:
# ============================================================
# RECHARGEMENT CORRECT FICHIER 2022
# ============================================================
# Le fichier 2022 a une structure particuliere :
# La ligne 0 contient les en-têtes mais les colonnes
# des candidats 2 a 12 sont nommees Unnamed car Excel
# a fusionne les cellules d en-tete.
# La solution est de recharger le fichier sans en-tete
# puis de reconstruire manuellement les noms de colonnes
# en repetant le pattern pour chaque candidat :
# N°Panneau, Sexe, Nom, Prenom, Voix, % Voix/Ins, % Voix/Exp
# ============================================================

# Recharger sans en-tete
df_2022_raw = pd.read_excel(
    RAW + "resultats-par-niveau-subcom-t1-france-entiere.xlsx",
    header=None
)

# Les 19 premieres colonnes ont des noms corrects
cols_base = [
    'Code du département', 'Libellé du département',
    'Code de la commune', 'Libellé de la commune',
    'Etat saisie', 'Inscrits', 'Abstentions', '% Abs/Ins',
    'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins',
    '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot',
    'Exprimés', '% Exp/Ins', '% Exp/Vot'
]

# Pour chaque candidat on a 7 colonnes qui se repetent
cols_candidat = ['N°Panneau', 'Sexe', 'Nom', 'Prénom',
                 'Voix', '% Voix/Ins', '% Voix/Exp']

# Construire la liste complete des noms de colonnes
# 12 candidats maximum en 2022
nouveaux_noms = cols_base.copy()
for i in range(12):
    if i == 0:
        nouveaux_noms += cols_candidat
    else:
        nouveaux_noms += [f"{c}.{i}" for c in cols_candidat]

# Appliquer les nouveaux noms au DataFrame
# On prend uniquement les colonnes qu on a nommees
df_2022_correct = df_2022_raw.copy()
df_2022_correct.columns = nouveaux_noms[:len(df_2022_raw.columns)]

# Supprimer la premiere ligne qui contient les anciens en-têtes
df_2022_correct = df_2022_correct.iloc[1:].reset_index(drop=True)

print("Fichier 2022 recharge correctement")
print("Dimensions :", df_2022_correct.shape)
print()
print("Verification colonnes Nom :")
cols_nom_2022 = [c for c in df_2022_correct.columns if c.startswith('Nom')]
print(cols_nom_2022)
print()
print("Verification premiere commune :")
for col in cols_nom_2022:
    print(f"  {col} : {df_2022_correct.iloc[0][col]}")

Fichier 2022 recharge correctement
Dimensions : (35245, 103)

Verification colonnes Nom :
['Nom', 'Nom.1', 'Nom.2', 'Nom.3', 'Nom.4', 'Nom.5', 'Nom.6', 'Nom.7', 'Nom.8', 'Nom.9', 'Nom.10', 'Nom.11']

Verification premiere commune :
  Nom : ARTHAUD
  Nom.1 : ROUSSEL
  Nom.2 : MACRON
  Nom.3 : LASSALLE
  Nom.4 : LE PEN
  Nom.5 : ZEMMOUR
  Nom.6 : MÉLENCHON
  Nom.7 : HIDALGO
  Nom.8 : JADOT
  Nom.9 : PÉCRESSE
  Nom.10 : POUTOU
  Nom.11 : DUPONT-AIGNAN


In [14]:
# ============================================================
# MISE A JOUR MAPPING ET REFILTRAGE 2022
# ============================================================
# On met a jour le mapping 2022 avec les noms exacts
# des candidats tels qu ils apparaissent dans le fichier
# incluant les accents comme MELENCHON et PECRESSE.
# On refait ensuite le filtrage des communes bretonnes
# sur le fichier 2022 correctement charge.
# ============================================================

# Mapping 2022 corrige avec accents exacts du fichier
MAPPING_2022 = {
    'ARTHAUD'        : 'Ext_gauche',
    'POUTOU'         : 'Ext_gauche',
    'MÉLENCHON'      : 'Gauche',
    'HIDALGO'        : 'Gauche',
    'JADOT'          : 'Gauche',
    'ROUSSEL'        : 'Gauche',
    'MACRON'         : 'Centre',
    'PÉCRESSE'       : 'Droite',
    'LE PEN'         : 'Ext_droite',
    'DUPONT-AIGNAN'  : 'Ext_droite',
    'ZEMMOUR'        : 'Ext_droite',
    'LASSALLE'       : 'Ext_droite',
}

# Refiltrage Bretagne sur le fichier 2022 correctement charge
df_2022_correct['Code du département'] = (
    df_2022_correct['Code du département']
    .astype(str).str.strip()
)

bretagne_2022 = df_2022_correct[
    df_2022_correct['Code du département'].isin(DEPTS_BRETAGNE)
].copy()
bretagne_2022['annee'] = 2022

print("Bretagne 2022 :", len(bretagne_2022), "communes")
print()

# Recalcul des scores 2022
cols_nom_2022   = [c for c in bretagne_2022.columns if c.startswith('Nom')]
cols_score_2022 = [c for c in bretagne_2022.columns if c.startswith('% Voix/Exp')]

scores_2022 = bretagne_2022.apply(
    lambda row: calculer_scores_v2(
        row, MAPPING_2022, cols_nom_2022, cols_score_2022
    ),
    axis=1
)

# Construction du DataFrame 2022
base_2022 = pd.DataFrame({
    'code_commune'    : bretagne_2022['Code de la commune'].astype(str).str.zfill(3),
    'nom_commune'     : bretagne_2022['Libellé de la commune'],
    'code_dept'       : bretagne_2022['Code du département'],
    'annee'           : 2022,
    'nb_inscrits'     : bretagne_2022['Inscrits'],
    'nb_votants'      : bretagne_2022['Votants'],
    'taux_abstention' : bretagne_2022['% Abs/Ins'],
}).reset_index(drop=True)

electoral_2022 = pd.concat(
    [base_2022, scores_2022.reset_index(drop=True)], axis=1
)

print("Verification scores 2022 :")
print(electoral_2022.head(3))

Bretagne 2022 : 1207 communes

Verification scores 2022 :
  code_commune nom_commune code_dept  annee nb_inscrits nb_votants  \
0          001    Allineuc        22   2022         455        373   
1          002       Andel        22   2022         930        775   
2          003    Aucaleuc        22   2022         731        594   

  taux_abstention  score_ext_gauche  score_gauche  score_centre  score_droite  \
0           18.02              2.19         23.50         26.23         10.66   
1           16.67              0.80         27.69         39.15          4.79   
2           18.74              2.41         35.23         20.45          1.37   

   score_ext_droite  
0             37.44  
1             27.57  
2             40.55  


In [15]:
# ============================================================
# FUSION 2017 ET 2022 + CALCUL DU GAGNANT
# ============================================================
# On fusionne les donnees electorales 2017 et 2022
# en un seul DataFrame avec pd.concat.
#
# Ensuite on calcule le gagnant pour chaque commune.
# Le gagnant est le groupe politique qui a obtenu
# le score le plus eleve dans cette commune cette annee.
#
# idxmax(axis=1) : trouve le nom de la colonne qui a
# la valeur maximale sur chaque ligne
# str.replace : nettoie le nom en enlevant score_
# pour garder uniquement le nom du groupe
# ============================================================

# Fusion 2017 et 2022
electoral = pd.concat(
    [electoral_2017, electoral_2022], axis=0
).reset_index(drop=True)

print("Dataset electoral fusionne :", electoral.shape)
print()

# Colonnes des scores
cols_scores = [
    'score_ext_gauche',
    'score_gauche',
    'score_centre',
    'score_droite',
    'score_ext_droite'
]

# Calcul du gagnant par commune et par annee
# idxmax trouve la colonne avec le score maximum
# str.replace enleve le prefixe score_ du nom
electoral['gagnant'] = (
    electoral[cols_scores]
    .idxmax(axis=1)
    .str.replace('score_', '', regex=False)
)

print("Verification avec gagnant :")
print(electoral[['code_commune', 'nom_commune', 'annee',
                  'score_centre', 'score_gauche',
                  'score_ext_droite', 'gagnant']].head(6))

print()
print("Distribution des gagnants :")
print(electoral['gagnant'].value_counts())

Dataset electoral fusionne : (2440, 12)

Verification avec gagnant :
  code_commune          nom_commune  annee  score_centre  score_gauche  \
0          001             Allineuc   2017         22.95         22.38   
1          002                Andel   2017         33.78         24.63   
2          003             Aucaleuc   2017         24.73         32.38   
3          004               Bégard   2017         24.21         34.78   
4          005  Belle-Isle-en-Terre   2017         22.88         35.62   
5          006               Berhet   2017         24.43         35.12   

   score_ext_droite     gagnant  
0             28.04  ext_droite  
1             20.86      centre  
2             29.89      gauche  
3             26.14      gauche  
4             24.18      gauche  
5             29.00      gauche  

Distribution des gagnants :
gagnant
ext_droite    959
centre        712
gauche        707
droite         62
Name: count, dtype: int64


In [16]:
# ============================================================
# CHARGEMENT INDICATEURS DEMOGRAPHIQUES
# ============================================================
# On charge les fichiers de population pour 2017 et 2021.
# Ces fichiers contiennent la population municipale
# par commune qui nous servira dans le dataset final.
#
# Le separateur est le point-virgule ; et non la virgule.
# C est une erreur courante avec les fichiers INSEE
# qui utilisent toujours le point-virgule comme separateur.
#
# DEPCOM ou CODCOM : code commune
# PMUN : population municipale officielle
# ============================================================

# Population 2017
df_pop_2017 = pd.read_csv(
    RAW + "Communes.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Population 2017 charge :", df_pop_2017.shape)
print("Colonnes :", df_pop_2017.columns.tolist())
print(df_pop_2017.head(3))

print()

# Population 2021
df_pop_2021 = pd.read_csv(
    RAW + "donnees_communes.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Population 2021 charge :", df_pop_2021.shape)
print("Colonnes :", df_pop_2021.columns.tolist())
print(df_pop_2021.head(3))

Population 2017 charge : (34995, 5)
Colonnes : ['DEPCOM', 'COM', 'PMUN', 'PCAP', 'PTOT']
  DEPCOM                        COM   PMUN  PCAP   PTOT
0  01001  L' Abergement-ClÃ©menciat    776    18    794
1  01002     L' Abergement-de-Varey    248     1    249
2  01004         AmbÃ©rieu-en-Bugey  14035   393  14428

Population 2021 charge : (34970, 11)
Colonnes : ['REG', 'RÃ©gion', 'DEP', 'CODARR', 'CODCAN', 'CODCOM', 'COM', 'Commune', 'PMUN', 'PCAP', 'PTOT']
   REG                RÃ©gion DEP  CODARR CODCAN  CODCOM    COM  \
0   84  Auvergne-RhÃ´ne-Alpes  01       2     08       1  01001   
1   84  Auvergne-RhÃ´ne-Alpes  01       1     01       2  01002   
2   84  Auvergne-RhÃ´ne-Alpes  01       1     01       4  01004   

                    Commune   PMUN  PCAP   PTOT  
0  L'Abergement-ClÃ©menciat    832    16    848  
1     L'Abergement-de-Varey    267     6    273  
2        AmbÃ©rieu-en-Bugey  14854   386  15240  


In [17]:
# ============================================================
# NETTOYAGE ET FILTRAGE POPULATION BRETAGNE
# ============================================================
# On nettoie les deux fichiers de population et on
# filtre uniquement les communes bretonnes.
#
# Le code commune dans ces fichiers est au format DEPCOM
# par exemple 22001 pour la premiere commune des
# Cotes-d-Armor. On doit extraire les 2 premiers
# caracteres pour avoir le code departement et filtrer.
#
# str.zfill(5) : complete le code commune avec des zeros
# a gauche pour avoir toujours 5 caracteres
# ex : 22001 et non 2201
#
# On garde uniquement les colonnes utiles :
# code_commune et population
# ============================================================

# Nettoyage population 2017
df_pop_2017['DEPCOM'] = df_pop_2017['DEPCOM'].astype(str).str.zfill(5)
df_pop_2017['dept']   = df_pop_2017['DEPCOM'].str[:2]

# Filtrage Bretagne 2017
pop_2017 = df_pop_2017[
    df_pop_2017['dept'].isin(DEPTS_BRETAGNE)
][['DEPCOM', 'PMUN']].copy()
pop_2017.columns = ['code_insee', 'population']
pop_2017['annee'] = 2017

print("Population 2017 Bretagne :", len(pop_2017), "communes")
print(pop_2017.head(3))

print()

# Nettoyage population 2021
df_pop_2021['DEPCOM'] = (
    df_pop_2021['DEP'].astype(str).str.zfill(2) +
    df_pop_2021['CODCOM'].astype(str).str.zfill(3)
)
df_pop_2021['dept'] = df_pop_2021['DEP'].astype(str).str.zfill(2)

# Filtrage Bretagne 2021
pop_2021 = df_pop_2021[
    df_pop_2021['dept'].isin(DEPTS_BRETAGNE)
][['DEPCOM', 'PMUN']].copy()
pop_2021.columns = ['code_insee', 'population']
pop_2021['annee'] = 2021

print("Population 2021 Bretagne :", len(pop_2021), "communes")
print(pop_2021.head(3))

Population 2017 Bretagne : 1208 communes
     code_insee  population  annee
7656      22001         604   2017
7657      22002        1124   2017
7658      22003         943   2017

Population 2021 Bretagne : 1207 communes
     code_insee  population  annee
7287      22001         596   2021
7288      22002        1156   2021
7289      22003         929   2021


In [18]:
# ============================================================
# CHARGEMENT SUPERFICIE ET DENSITE
# ============================================================
# On charge le fichier communes-france-2022.csv qui contient
# la superficie en km2 et la densite de population
# pour chaque commune de France.
#
# Ces donnees sont fixes dans le temps donc on les utilise
# pour les deux annees 2017 et 2022.
#
# On affiche les colonnes pour identifier les bonnes
# colonnes a utiliser.
# ============================================================

df_geo = pd.read_csv(
    RAW + "communes-france-2022.csv",
    sep=",",
    encoding="utf-8",
    low_memory=False
)
print("Fichier geo charge :", df_geo.shape)
print("Colonnes :", df_geo.columns.tolist())
print()
print("5 premieres lignes :")
print(df_geo.head(3))

Fichier geo charge : (35010, 39)
Colonnes : ['Unnamed: 0', 'code_insee', 'nom_standard', 'nom_sans_pronom', 'nom_a', 'nom_de', 'nom_sans_accent', 'nom_standard_majuscule', 'typecom', 'typecom_texte', 'reg_code', 'reg_nom', 'dep_code', 'dep_nom', 'canton_code', 'canton_nom', 'epci_code', 'epci_nom', 'academie_code', 'academie_nom', 'code_postal', 'codes_postaux', 'zone_emploi', 'code_insee_centre_zone_emploi', 'population', 'superficie_hectare', 'superficie_km2', 'densite', 'altitude_moyenne', 'altitude_minimale', 'altitude_maximale', 'latitude_mairie', 'longitude_mairie', 'latitude_centre', 'longitude_centre', 'grille_densite', 'gentile', 'url_wikipedia', 'url_villedereve']

5 premieres lignes :
   Unnamed: 0 code_insee             nom_standard        nom_sans_pronom  \
0           0      01001  L'Abergement-Clémenciat  Abergement-Clémenciat   
1           1      01002    L'Abergement-de-Varey    Abergement-de-Varey   
2           2      01004        Ambérieu-en-Bugey      Ambérieu-en-

In [19]:
# ============================================================
# FILTRAGE GEO BRETAGNE
# ============================================================
# On filtre uniquement les communes bretonnes du fichier geo
# et on garde uniquement les colonnes utiles :
# code_insee : code commune sur 5 caracteres
# superficie_km2 : superficie de la commune en km2
# densite : densite de population par km2
#
# dep_code : code departement pour filtrer la Bretagne
# On convertit dep_code en string et on filtre
# sur les 4 departements bretons.
# ============================================================

# Filtrage Bretagne
df_geo['dep_code'] = df_geo['dep_code'].astype(str).str.zfill(2)

geo_bretagne = df_geo[
    df_geo['dep_code'].isin(DEPTS_BRETAGNE)
][['code_insee', 'superficie_km2', 'densite']].copy()

geo_bretagne['code_insee'] = geo_bretagne['code_insee'].astype(str).str.zfill(5)

print("Geo Bretagne :", len(geo_bretagne), "communes")
print(geo_bretagne.head(3))

Geo Bretagne : 1208 communes
     code_insee  superficie_km2  densite
7295      22001              24     25.1
7296      22002              12     92.1
7297      22003               6    147.8


In [20]:
# ============================================================
# CHARGEMENT EMPLOI ET EDUCATION
# ============================================================
# On charge le fichier base-cc-emploi-pop-active-2021.CSV
# qui contient les donnees d emploi et d education
# par commune pour 2015 et 2021.
#
# Les colonnes qui nous interessent sont :
# CODGEO : code commune sur 5 caracteres
#
# Pour le chomage :
# P15_CHOM1564 : chomeurs 15-64 ans en 2015
# P15_ACT1564  : actifs 15-64 ans en 2015
# P21_CHOM1564 : chomeurs 15-64 ans en 2021
# P21_ACT1564  : actifs 15-64 ans en 2021
#
# Pour l education :
# P15_NSCOL15P_SUP   : diplomes du superieur en 2015
# P15_NSCOL15P       : total non scolarises en 2015
# P21_NSCOL15P_SUP   : diplomes du superieur en 2021
# P21_NSCOL15P       : total non scolarises en 2021
#
# On affiche les colonnes pour verifier les noms exacts
# ============================================================

df_emploi = pd.read_csv(
    RAW + "base-cc-emploi-pop-active-2021.CSV",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Emploi charge :", df_emploi.shape)
print()

# Afficher les colonnes qui contiennent CHOM ACT SUP
cols_utiles = [c for c in df_emploi.columns
               if any(x in c for x in ['CODGEO', 'CHOM', 'ACT', 'SUP', 'NSCOL'])]
print("Colonnes utiles :")
for c in cols_utiles:
    print(" -", c)

Emploi charge : (34963, 354)

Colonnes utiles :
 - CODGEO
 - P21_ACT1564
 - P21_ACT1524
 - P21_ACT2554
 - P21_ACT5564
 - P21_HACT1564
 - P21_HACT1524
 - P21_HACT2554
 - P21_HACT5564
 - P21_FACT1564
 - P21_FACT1524
 - P21_FACT2554
 - P21_FACT5564
 - P21_ACTOCC1564
 - P21_ACTOCC1524
 - P21_ACTOCC2554
 - P21_ACTOCC5564
 - P21_HACTOCC1564
 - P21_HACTOCC1524
 - P21_HACTOCC2554
 - P21_HACTOCC5564
 - P21_FACTOCC1564
 - P21_FACTOCC1524
 - P21_FACTOCC2554
 - P21_FACTOCC5564
 - P21_CHOM1564
 - P21_CHOM1524
 - P21_CHOM2554
 - P21_CHOM5564
 - P21_CHOM_DIPLMIN
 - P21_CHOM_BEPC
 - P21_CHOM_CAPBEP
 - P21_CHOM_BAC
 - P21_CHOM_SUP2
 - P21_CHOM_SUP34
 - P21_CHOM_SUP5
 - P21_ACT_DIPLMIN
 - P21_ACT_BEPC
 - P21_ACT_CAPBEP
 - P21_ACT_BAC
 - P21_ACT_SUP2
 - P21_ACT_SUP34
 - P21_ACT_SUP5
 - P21_INACT1564
 - P21_AINACT1564
 - C21_ACT1564
 - C21_ACT1564_CS1
 - C21_ACT1564_CS2
 - C21_ACT1564_CS3
 - C21_ACT1564_CS4
 - C21_ACT1564_CS5
 - C21_ACT1564_CS6
 - C21_ACTOCC1564
 - C21_ACTOCC1564_CS1
 - C21_ACTOCC1564_CS2

In [21]:
# ============================================================
# IDENTIFIER COLONNES CHOMAGE ET EDUCATION
# ============================================================
# On cherche precisement les colonnes disponibles pour
# le chomage et l education dans ce fichier.
# On affiche toutes les colonnes qui contiennent
# les mots cles CHOM et NSCOL pour trouver les
# noms exacts a utiliser dans notre ETL.
# ============================================================

print("Colonnes contenant CHOM :")
cols_chom = [c for c in df_emploi.columns if 'CHOM' in c]
for c in cols_chom:
    print(" -", c)

print()
print("Colonnes contenant NSCOL :")
cols_nscol = [c for c in df_emploi.columns if 'NSCOL' in c]
for c in cols_nscol:
    print(" -", c)

print()
print("Colonnes contenant DIPL :")
cols_dipl = [c for c in df_emploi.columns if 'DIPL' in c]
for c in cols_dipl:
    print(" -", c)

Colonnes contenant CHOM :
 - P21_CHOM1564
 - P21_CHOM1524
 - P21_CHOM2554
 - P21_CHOM5564
 - P21_CHOM_DIPLMIN
 - P21_CHOM_BEPC
 - P21_CHOM_CAPBEP
 - P21_CHOM_BAC
 - P21_CHOM_SUP2
 - P21_CHOM_SUP34
 - P21_CHOM_SUP5
 - P21_HCHOM1564
 - P21_HCHOM1524
 - P21_HCHOM2554
 - P21_HCHOM5564
 - P21_FCHOM1564
 - P21_FCHOM1524
 - P21_FCHOM2554
 - P21_FCHOM5564
 - P15_CHOM1564
 - P15_HCHOM1564
 - P15_HCHOM1524
 - P15_HCHOM2554
 - P15_HCHOM5564
 - P15_FCHOM1564
 - P15_FCHOM1524
 - P15_FCHOM2554
 - P15_FCHOM5564
 - P10_CHOM1564
 - P10_HCHOM1564
 - P10_HCHOM1524
 - P10_HCHOM2554
 - P10_HCHOM5564
 - P10_FCHOM1564
 - P10_FCHOM1524
 - P10_FCHOM2554
 - P10_FCHOM5564

Colonnes contenant NSCOL :

Colonnes contenant DIPL :
 - P21_CHOM_DIPLMIN
 - P21_ACT_DIPLMIN


In [22]:
# ============================================================
# EXTRACTION EMPLOI ET EDUCATION 
# ============================================================
# Apres verification le fichier contient :
#
# Pour le chomage :
# P15_CHOM1564 et P15_ACT1564 pour 2015 proxy 2017
# P21_CHOM1564 et P21_ACT1564 pour 2021
#
# Pour les diplomes du superieur 2021 uniquement :
# P21_ACT_SUP2  : diplomes bac+2
# P21_ACT_SUP34 : diplomes bac+3 et bac+4
# P21_ACT_SUP5  : diplomes bac+5 et plus
# P21_ACT1564   : total actifs 15-64 ans
#
# Pour sans diplome 2021 uniquement :
# P21_ACT_DIPLMIN : actifs sans diplome ou brevet
# P21_ACT1564     : total actifs 15-64 ans
#
# Pour 2017 on utilisera les memes valeurs education
# que 2021 car c est la seule donnee disponible.
# On le justifiera dans le rapport.

# Note : les colonnes education (pct_diplomes_sup, pct_sans_diplome)
# utilisent uniquement les donnees P21_ car le fichier INSEE
# ne fournit pas les colonnes equivalentes pour 2015.
# Cette limitation sera justifiee dans le rapport.
# ============================================================

# Filtrage Bretagne
df_emploi['CODGEO'] = df_emploi['CODGEO'].astype(str).str.zfill(5)
df_emploi['dept']   = df_emploi['CODGEO'].str[:2]

emploi_bret = df_emploi[
    df_emploi['dept'].isin(DEPTS_BRETAGNE)
].copy()

print("Emploi Bretagne :", len(emploi_bret), "communes")
print()

# Calcul taux chomage 2015 proxy 2017
emploi_bret['taux_chomage_2017'] = np.where(
    emploi_bret['P15_ACT1564'] > 0,
    (emploi_bret['P15_CHOM1564'] / emploi_bret['P15_ACT1564']) * 100,
    0
)

# Calcul taux chomage 2021
emploi_bret['taux_chomage_2021'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    (emploi_bret['P21_CHOM1564'] / emploi_bret['P21_ACT1564']) * 100,
    0
)

# Calcul pct diplomes superieur 2021
# On additionne bac+2, bac+3/4 et bac+5 et plus
emploi_bret['pct_diplomes_sup'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    ((emploi_bret['P21_ACT_SUP2'] +
      emploi_bret['P21_ACT_SUP34'] +
      emploi_bret['P21_ACT_SUP5']) /
      emploi_bret['P21_ACT1564']) * 100,
    0
)

# Calcul pct sans diplome 2021
emploi_bret['pct_sans_diplome'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    (emploi_bret['P21_ACT_DIPLMIN'] / emploi_bret['P21_ACT1564']) * 100,
    0
)

# Garder uniquement les colonnes utiles
emploi_final = emploi_bret[[
    'CODGEO',
    'taux_chomage_2017',
    'taux_chomage_2021',
    'pct_diplomes_sup',
    'pct_sans_diplome'
]].copy()

emploi_final.columns = [
    'code_insee',
    'taux_chomage_2017',
    'taux_chomage_2021',
    'pct_diplomes_sup',
    'pct_sans_diplome'
]

print("Emploi et education Bretagne :", len(emploi_final), "communes")
print()
print("Verification :")
print(emploi_final.head(3))
print()
print("Statistiques :")
print(emploi_final.describe().round(2))

Emploi Bretagne : 1206 communes

Emploi et education Bretagne : 1206 communes

Verification :
     code_insee  taux_chomage_2017  taux_chomage_2021  pct_diplomes_sup  \
7285      22001           7.450980           7.117357         29.883846   
7286      22002           6.431711           6.427328         39.727704   
7287      22003           9.386241           8.399250         25.514695   

      pct_sans_diplome  
7285          5.707780  
7286          3.966657  
7287          7.130730  

Statistiques :
       taux_chomage_2017  taux_chomage_2021  pct_diplomes_sup  \
count            1206.00            1206.00           1206.00   
mean               10.63               9.15             35.80   
std                 3.15               3.23              9.26   
min                 2.68               2.31             15.26   
25%                 8.47               6.89             29.53   
50%                10.16               8.62             34.60   
75%                12.43          

In [23]:
# ============================================================
# CHERCHER COLONNES EDUCATION DISPONIBLES
# ============================================================
# La colonne P15_NSCOL15P n existe pas dans ce fichier.
# On cherche toutes les colonnes disponibles qui
# contiennent SUP et NSCOL pour trouver les noms
# exacts des colonnes education disponibles.
# ============================================================

print("Colonnes contenant NSCOL :")
cols_nscol = [c for c in df_emploi.columns if 'NSCOL' in c]
for c in cols_nscol:
    print(" -", c)

print()
print("Colonnes contenant SUP :")
cols_sup = [c for c in df_emploi.columns if 'SUP' in c]
for c in cols_sup:
    print(" -", c)

print()
print("Colonnes contenant ACT_D :")
cols_actd = [c for c in df_emploi.columns if 'ACT_D' in c]
for c in cols_actd:
    print(" -", c)

Colonnes contenant NSCOL :

Colonnes contenant SUP :
 - P21_CHOM_SUP2
 - P21_CHOM_SUP34
 - P21_CHOM_SUP5
 - P21_ACT_SUP2
 - P21_ACT_SUP34
 - P21_ACT_SUP5

Colonnes contenant ACT_D :
 - P21_ACT_DIPLMIN


In [24]:
# ============================================================
# EXTRACTION EMPLOI ET EDUCATION CORRIGEE
# ============================================================
# Apres verification le fichier contient :
#
# Pour le chomage :
# P15_CHOM1564 et P15_ACT1564 pour 2015 (proxy 2017)
# P21_CHOM1564 et P21_ACT1564 pour 2021
#
# Pour les diplomes du superieur 2021 uniquement :
# P21_ACT_SUP2  : diplomes bac+2
# P21_ACT_SUP34 : diplomes bac+3 et bac+4
# P21_ACT_SUP5  : diplomes bac+5 et plus
# P21_ACT1564   : total actifs 15-64 ans
#
# Pour sans diplome 2021 uniquement :
# P21_ACT_DIPLMIN : actifs sans diplome ou brevet
# P21_ACT1564     : total actifs 15-64 ans
#
# Pour 2017 on utilisera les memes valeurs education
# que 2021 car c est la seule donnee disponible.
# On le justifiera dans le rapport.

# Note : les colonnes education (pct_diplomes_sup, pct_sans_diplome)
# utilisent uniquement les donnees P21_ car le fichier INSEE
# ne fournit pas les colonnes equivalentes pour 2015.
# Cette limitation sera justifiee dans le rapport.
# ============================================================

# Calcul taux chomage 2015 proxy 2017
emploi_bret['taux_chomage_2017'] = np.where(
    emploi_bret['P15_ACT1564'] > 0,
    (emploi_bret['P15_CHOM1564'] / emploi_bret['P15_ACT1564']) * 100,
    0
)

# Calcul taux chomage 2021
emploi_bret['taux_chomage_2021'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    (emploi_bret['P21_CHOM1564'] / emploi_bret['P21_ACT1564']) * 100,
    0
)

# Calcul pct diplomes superieur 2021
# On additionne bac+2, bac+3/4 et bac+5 et plus
emploi_bret['pct_diplomes_sup'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    ((emploi_bret['P21_ACT_SUP2'] +
      emploi_bret['P21_ACT_SUP34'] +
      emploi_bret['P21_ACT_SUP5']) /
      emploi_bret['P21_ACT1564']) * 100,
    0
)

# Calcul pct sans diplome 2021
emploi_bret['pct_sans_diplome'] = np.where(
    emploi_bret['P21_ACT1564'] > 0,
    (emploi_bret['P21_ACT_DIPLMIN'] / emploi_bret['P21_ACT1564']) * 100,
    0
)

# Garder uniquement les colonnes utiles
emploi_final = emploi_bret[[
    'CODGEO',
    'taux_chomage_2017',
    'taux_chomage_2021',
    'pct_diplomes_sup',
    'pct_sans_diplome'
]].copy()

emploi_final.columns = [
    'code_insee',
    'taux_chomage_2017',
    'taux_chomage_2021',
    'pct_diplomes_sup',
    'pct_sans_diplome'
]

print("Emploi et education Bretagne :", len(emploi_final), "communes")
print()
print("Verification :")
print(emploi_final.head(3))
print()
print("Statistiques :")
print(emploi_final.describe().round(2))

Emploi et education Bretagne : 1206 communes

Verification :
     code_insee  taux_chomage_2017  taux_chomage_2021  pct_diplomes_sup  \
7285      22001           7.450980           7.117357         29.883846   
7286      22002           6.431711           6.427328         39.727704   
7287      22003           9.386241           8.399250         25.514695   

      pct_sans_diplome  
7285          5.707780  
7286          3.966657  
7287          7.130730  

Statistiques :
       taux_chomage_2017  taux_chomage_2021  pct_diplomes_sup  \
count            1206.00            1206.00           1206.00   
mean               10.63               9.15             35.80   
std                 3.15               3.23              9.26   
min                 2.68               2.31             15.26   
25%                 8.47               6.89             29.53   
50%                10.16               8.62             34.60   
75%                12.43              10.76             41.08   
ma

In [25]:
# ============================================================
# CHARGEMENT REVENU MEDIAN
# ============================================================
# On charge les fichiers Filosofi qui contiennent
# le revenu median par commune.
#
# cc_filosofi_2017_COM.CSV : revenu median 2017
# FILO2021_DEC_COM.csv     : revenu median 2021
#
# Le revenu median est le revenu au dessus et en dessous
# duquel se trouvent 50% des habitants. C est un meilleur
# indicateur que la moyenne car il n est pas fausse
# par les tres hauts revenus.
#
# On affiche les colonnes pour identifier les bonnes
# colonnes a utiliser dans chaque fichier.
# ============================================================

# Revenu median 2017
df_rev_2017 = pd.read_csv(
    RAW + "cc_filosofi_2017_COM.CSV",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Revenu 2017 charge :", df_rev_2017.shape)
print("Colonnes :", df_rev_2017.columns.tolist())
print()

# Revenu median 2021
df_rev_2021 = pd.read_csv(
    RAW + "FILO2021_DEC_COM.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Revenu 2021 charge :", df_rev_2021.shape)
print("Colonnes :", df_rev_2021.columns.tolist())

Revenu 2017 charge : (34931, 28)
Colonnes : ['CODGEO', 'NBMENFISC17', 'NBPERSMENFISC17', 'MED17', 'PIMP17', 'TP6017', 'TP60AGE117', 'TP60AGE217', 'TP60AGE317', 'TP60AGE417', 'TP60AGE517', 'TP60AGE617', 'TP60TOL117', 'TP60TOL217', 'PACT17', 'PTSA17', 'PCHO17', 'PBEN17', 'PPEN17', 'PPAT17', 'PPSOC17', 'PPFAM17', 'PPMINI17', 'PPLOGT17', 'PIMPOT17', 'D117', 'D917', 'RD17']

Revenu 2021 charge : (34929, 572)
Colonnes : ['CODGEO', 'NBMEN21', 'NBPERS21', 'NBUC21', 'PMIMP21', 'Q121', 'Q221', 'Q321', 'Q3_Q1', 'D121', 'D221', 'D321', 'D421', 'D621', 'D721', 'D821', 'D921', 'RD', 'S80S2021', 'GI21', 'PACT21', 'PTSA21', 'PCHO21', 'PBEN21', 'PPEN21', 'PAUT21', 'AGE1Q121', 'AGE1Q221', 'AGE1Q321', 'AGE1Q3_Q1', 'AGE1D121', 'AGE1D221', 'AGE1D321', 'AGE1D421', 'AGE1D621', 'AGE1D721', 'AGE1D821', 'AGE1D921', 'AGE1RD', 'AGE1S80S2021', 'AGE1GI21', 'AGE1PACT21', 'AGE1PTSA21', 'AGE1PCHO21', 'AGE1PBEN21', 'AGE1PPEN21', 'AGE1PAUT21', 'AGE2Q121', 'AGE2Q221', 'AGE2Q321', 'AGE2Q3_Q1', 'AGE2D121', 'AGE2D221', 'AGE

In [26]:
# ============================================================
# IDENTIFIER COLONNE REVENU MEDIAN 2021
# ============================================================
# On cherche la colonne revenu median dans le fichier 2021
# En 2017 elle s appelait MED17
# En 2021 elle devrait s appeler MED21 ou Q221
# On affiche toutes les colonnes contenant MED ou Q2
# pour trouver le bon nom.
# ============================================================

print("Colonnes contenant MED :")
cols_med = [c for c in df_rev_2021.columns if 'MED' in c]
for c in cols_med:
    print(" -", c)

print()
print("Colonnes contenant Q2 :")
cols_q2 = [c for c in df_rev_2021.columns if 'Q2' in c]
for c in cols_q2:
    print(" -", c)

print()
print("5 premieres lignes fichier 2021 :")
print(df_rev_2021.head(3))

Colonnes contenant MED :

Colonnes contenant Q2 :
 - Q221
 - AGE1Q221
 - AGE2Q221
 - AGE3Q221
 - AGE4Q221
 - AGE5Q221
 - AGE6Q221
 - TME1Q221
 - TME2Q221
 - TME3Q221
 - TME4Q221
 - TME5Q221
 - TOL1Q221
 - TOL2Q221
 - TLD2Q221
 - TLD3Q221
 - TYM1Q221
 - TYM2Q221
 - TYM3Q221
 - TYM4Q221
 - TYM5Q221
 - TYM6Q221
 - OPR1Q221
 - OPR2Q221
 - OPR3Q221
 - OPR4Q221
 - OPR5Q221

5 premieres lignes fichier 2021 :
  CODGEO NBMEN21 NBPERS21   NBUC21 PMIMP21   Q121   Q221   Q321  Q3_Q1  D121  \
0  01001     346      895    590,8       s      s  26410      s      s     s   
1  01002     115      266    181,0       s      s  24950      s      s     s   
2  01004    6861    15103  10406,5    50,0  13180  20940  29260  16080  7070   

   ... OPR5D921 OPR5RD OPR5S80S2021 OPR5GI21 OPR5PACT21 OPR5PTSA21 OPR5PCHO21  \
0  ...        s      s            s        s          s          s          s   
1  ...        s      s            s        s          s          s          s   
2  ...        s      s         

In [27]:
# ============================================================
# EXTRACTION REVENU MEDIAN BRETAGNE
# ============================================================
# On extrait le revenu median pour les communes bretonnes
# depuis les deux fichiers Filosofi.
#
# MED17 : revenu median 2017 dans cc_filosofi_2017_COM
# Q221  : revenu median 2021 dans FILO2021_DEC_COM
#
# Attention : certaines petites communes n ont pas de
# donnees de revenu car INSEE ne publie pas les donnees
# quand la commune a moins de 500 habitants fiscaux.
# Ces valeurs manquantes sont notees s dans le fichier.
# On les remplace par NaN avec pd.to_numeric et errors=coerce
# qui convertit les valeurs non numeriques en NaN.
# ============================================================

# Revenu median 2017 Bretagne
df_rev_2017['CODGEO'] = df_rev_2017['CODGEO'].astype(str).str.zfill(5)
df_rev_2017['dept']   = df_rev_2017['CODGEO'].str[:2]

rev_2017 = df_rev_2017[
    df_rev_2017['dept'].isin(DEPTS_BRETAGNE)
][['CODGEO', 'MED17']].copy()

rev_2017.columns = ['code_insee', 'revenu_median_2017']
rev_2017['revenu_median_2017'] = pd.to_numeric(
    rev_2017['revenu_median_2017'], errors='coerce'
)

print("Revenu median 2017 Bretagne :", len(rev_2017), "communes")
print("Valeurs manquantes :", rev_2017['revenu_median_2017'].isna().sum())
print(rev_2017.head(3))

print()

# Revenu median 2021 Bretagne
df_rev_2021['CODGEO'] = df_rev_2021['CODGEO'].astype(str).str.zfill(5)
df_rev_2021['dept']   = df_rev_2021['CODGEO'].str[:2]

rev_2021 = df_rev_2021[
    df_rev_2021['dept'].isin(DEPTS_BRETAGNE)
][['CODGEO', 'Q221']].copy()

rev_2021.columns = ['code_insee', 'revenu_median_2021']
rev_2021['revenu_median_2021'] = pd.to_numeric(
    rev_2021['revenu_median_2021'], errors='coerce'
)

print("Revenu median 2021 Bretagne :", len(rev_2021), "communes")
print("Valeurs manquantes :", rev_2021['revenu_median_2021'].isna().sum())
print(rev_2021.head(3))

Revenu median 2017 Bretagne : 1206 communes
Valeurs manquantes : 3
     code_insee  revenu_median_2017
7294      22001             19360.0
7295      22002             21310.0
7296      22003             20530.0

Revenu median 2021 Bretagne : 1207 communes
Valeurs manquantes : 5
     code_insee  revenu_median_2021
7292      22001             21360.0
7293      22002             24680.0
7294      22003             22150.0


In [28]:
# ============================================================
# CHARGEMENT CRIMINALITE
# ============================================================
# On charge le fichier de la base communale de delinquance
# du SSMSI qui contient le taux de criminalite par commune.
#
# Ce fichier contient plusieurs types d infractions
# par commune et par annee. On va aggreger tous les
# types d infractions pour obtenir un taux global
# de criminalite par commune.
#
# Les colonnes utiles sont :
# CODGEO_2025 : code commune
# annee       : annee de l infraction
# taux_pour_mille : taux d infractions pour 1000 habitants
#
# On filtre sur les annees 2017 et 2021 pour correspondre
# a nos deux periodes d analyse.
# ============================================================

# Trouver le bon nom du fichier delinquance
fichier_crime = [f for f in os.listdir(RAW) if 'donnee' in f.lower()][0]
print("Fichier criminalite :", fichier_crime)
print()

df_crime = pd.read_csv(
    RAW + fichier_crime,
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Criminalite charge :", df_crime.shape)
print("Colonnes :", df_crime.columns.tolist())
print()
print("Annees disponibles :", sorted(df_crime['annee'].unique()))

Fichier criminalite : donnee-data.gouv-2025-geographie2025-produit-le2026-02-03 (1).csv

Criminalite charge : (5238000, 13)
Colonnes : ['CODGEO_2025', 'annee', 'indicateur', 'unite_de_compte', 'nombre', 'taux_pour_mille', 'est_diffuse', 'insee_pop', 'insee_pop_millesime', 'insee_log', 'insee_log_millesime', 'complement_info_nombre', 'complement_info_taux']

Annees disponibles : [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [29]:
# ============================================================
# EXTRACTION CRIMINALITE BRETAGNE
# ============================================================
# Le fichier contient 5 millions de lignes car il y a
# plusieurs types d infractions par commune par annee.
# On va aggreger tous les types d infractions pour
# obtenir un taux global de criminalite par commune.
#
# La strategie est la suivante :
# 1. Filtrer uniquement les annees 2017 et 2021
# 2. Filtrer uniquement les communes bretonnes
#    en utilisant les 2 premiers caracteres du code commune
# 3. Convertir taux_pour_mille en numerique
#    car certaines valeurs sont marquees NA dans le fichier
# 4. Grouper par commune et par annee
#    et faire la somme des taux de tous les types
#    d infractions pour obtenir un taux global
# ============================================================

# Filtrage annees 2017 et 2021
df_crime_filtre = df_crime[
    df_crime['annee'].isin([2017, 2021])
].copy()

print("Apres filtrage annees :", len(df_crime_filtre), "lignes")

# Conversion code commune en string
df_crime_filtre['CODGEO_2025'] = (
    df_crime_filtre['CODGEO_2025'].astype(str).str.zfill(5)
)

# Extraction code departement
df_crime_filtre['dept'] = df_crime_filtre['CODGEO_2025'].str[:2]

# Filtrage Bretagne
df_crime_bret = df_crime_filtre[
    df_crime_filtre['dept'].isin(DEPTS_BRETAGNE)
].copy()

print("Apres filtrage Bretagne :", len(df_crime_bret), "lignes")

# Conversion taux_pour_mille en numerique
df_crime_bret['taux_pour_mille'] = pd.to_numeric(
    df_crime_bret['taux_pour_mille'], errors='coerce'
)

# Aggregation par commune et par annee
# On fait la moyenne des taux pour avoir un taux global
crime_agg = df_crime_bret.groupby(
    ['CODGEO_2025', 'annee']
)['taux_pour_mille'].mean().reset_index()

crime_agg.columns = ['code_insee', 'annee', 'taux_criminalite']

print()
print("Criminalite aggregee Bretagne :", len(crime_agg), "lignes")
print()
print("Verification :")
print(crime_agg.head(6))

Apres filtrage annees : 1047600 lignes
Apres filtrage Bretagne : 36060 lignes

Criminalite aggregee Bretagne : 2404 lignes

Verification :
  code_insee  annee  taux_criminalite
0      22001   2017               NaN
1      22001   2021               NaN
2      22002   2017               NaN
3      22002   2021               NaN
4      22003   2017               NaN
5      22003   2021               NaN


In [30]:
# ============================================================
# DIAGNOSTIC DONNEES CRIMINALITE
# ============================================================
# Les taux sont tous NaN ce qui signifie que les donnees
# ne sont pas diffusees pour les petites communes.
# On examine le contenu du fichier pour comprendre
# quelles donnees sont reellement disponibles.
# On regarde la colonne est_diffuse et les valeurs
# reelles de taux_pour_mille pour quelques communes.
# ============================================================

# Verifier la colonne est_diffuse
print("Valeurs de est_diffuse :")
print(df_crime_bret['est_diffuse'].value_counts())

print()

# Verifier les valeurs brutes de taux_pour_mille
print("Exemples de valeurs taux_pour_mille :")
print(df_crime_bret['taux_pour_mille'].value_counts().head(10))

print()

# Regarder les lignes ou est_diffuse est diff
df_diff = df_crime_bret[df_crime_bret['est_diffuse'] == 'diff'].head(10)
print("Lignes diffusees :")
print(df_diff[['CODGEO_2025', 'annee', 'indicateur',
               'taux_pour_mille', 'est_diffuse']].head(10))

Valeurs de est_diffuse :
est_diffuse
ndiff    20376
diff     15684
Name: count, dtype: int64

Exemples de valeurs taux_pour_mille :
Series([], Name: count, dtype: int64)

Lignes diffusees :
       CODGEO_2025  annee                               indicateur  \
632955       22001   2017      Violences physiques intrafamiliales   
632958       22001   2017                          Vols avec armes   
632959       22001   2017                  Vols violents sans arme   
632962       22001   2017                        Vols de vÃ©hicule   
632964       22001   2017        Vols d'accessoires sur vÃ©hicules   
632967       22001   2017              Usage de stupÃ©fiants (AFD)   
632968       22001   2017                   Trafic de stupÃ©fiants   
632971       22002   2017  Violences physiques hors cadre familial   
632973       22002   2017                          Vols avec armes   
632974       22002   2017                  Vols violents sans arme   

        taux_pour_mille est_diffuse  
6

In [31]:
# ============================================================
# DIAGNOSTIC COLONNES VALEURS REELLES
# ============================================================
# Les valeurs de taux_pour_mille sont NaN meme pour
# les lignes diffusees. On examine toutes les colonnes
# numeriques pour trouver ou sont les vraies valeurs.
# On regarde aussi les colonnes complement_info qui
# pourraient contenir les taux reels.
# ============================================================

# Regarder une ligne complete pour une commune connue
print("Exemple complet premiere ligne diffusee :")
ligne = df_crime_bret[df_crime_bret['est_diffuse'] == 'diff'].iloc[0]
for col in df_crime_bret.columns:
    print(f"  {col} : {ligne[col]}")

print()

# Verifier complement_info_taux
print("Exemples complement_info_taux :")
print(df_crime_bret[
    df_crime_bret['est_diffuse'] == 'diff'
]['complement_info_taux'].head(10))

print()

# Verifier complement_info_nombre
print("Exemples complement_info_nombre :")
print(df_crime_bret[
    df_crime_bret['est_diffuse'] == 'diff'
]['complement_info_nombre'].head(10))

Exemple complet premiere ligne diffusee :
  CODGEO_2025 : 22001
  annee : 2017
  indicateur : Violences physiques intrafamiliales
  unite_de_compte : Victime
  nombre : 0.0
  taux_pour_mille : nan
  est_diffuse : diff
  insee_pop : 604
  insee_pop_millesime : 2017
  insee_log : 319
  insee_log_millesime : 2017
  complement_info_nombre : nan
  complement_info_taux : nan
  dept : 22

Exemples complement_info_taux :
632955    NaN
632958    NaN
632959    NaN
632962    NaN
632964    NaN
632967    NaN
632968    NaN
632971    NaN
632973    NaN
632974    NaN
Name: complement_info_taux, dtype: object

Exemples complement_info_nombre :
632955    NaN
632958    NaN
632959    NaN
632962    NaN
632964    NaN
632967    NaN
632968    NaN
632971    NaN
632973    NaN
632974    NaN
Name: complement_info_nombre, dtype: object


In [32]:
# ============================================================
# CALCUL TAUX CRIMINALITE
# ============================================================
# Le taux_pour_mille est NaN quand le nombre est 0
# ou quand les donnees ne sont pas diffusees.
# On va calculer nous memes le taux de criminalite
# en divisant le nombre total d infractions par
# la population de la commune multiplie par 1000.
#
# taux = (nombre_total_infractions / population) x 1000
#
# On garde uniquement les lignes diffusees car les
# lignes ndiff ne sont pas fiables.
# On remplace les NaN de nombre par 0 car une valeur
# manquante signifie souvent 0 infraction dans ce fichier.
# ============================================================

# Garder uniquement les lignes diffusees
df_crime_diff = df_crime_bret[
    df_crime_bret['est_diffuse'] == 'diff'
].copy()

print("Lignes diffusees Bretagne :", len(df_crime_diff))

# Remplacer NaN dans nombre par 0
df_crime_diff['nombre'] = pd.to_numeric(
    df_crime_diff['nombre'], errors='coerce'
).fillna(0)

# Remplacer NaN dans insee_pop par 0
df_crime_diff['insee_pop'] = pd.to_numeric(
    df_crime_diff['insee_pop'], errors='coerce'
).fillna(0)

# Aggreger le nombre total d infractions par commune et annee
crime_total = df_crime_diff.groupby(
    ['CODGEO_2025', 'annee', 'insee_pop']
)['nombre'].sum().reset_index()

print()
print("Verification agregation :")
print(crime_total.head(6))

# Calcul du taux pour mille habitants
crime_total['taux_criminalite'] = np.where(
    crime_total['insee_pop'] > 0,
    (crime_total['nombre'] / crime_total['insee_pop']) * 1000,
    0
)

# Garder uniquement les colonnes utiles
crime_final = crime_total[
    ['CODGEO_2025', 'annee', 'taux_criminalite']
].copy()
crime_final.columns = ['code_insee', 'annee', 'taux_criminalite']

print()
print("Taux criminalite Bretagne :", len(crime_final), "lignes")
print()
print("Verification :")
print(crime_final.head(6))
print()
print("Statistiques :")
print(crime_final['taux_criminalite'].describe().round(2))

Lignes diffusees Bretagne : 15684

Verification agregation :
  CODGEO_2025  annee  insee_pop  nombre
0       22001   2017        604     0.0
1       22001   2021        596     0.0
2       22002   2017       1124     0.0
3       22002   2021       1156     0.0
4       22003   2017        943     0.0
5       22003   2021        929     0.0

Taux criminalite Bretagne : 2404 lignes

Verification :
  code_insee  annee  taux_criminalite
0      22001   2017               0.0
1      22001   2021               0.0
2      22002   2017               0.0
3      22002   2021               0.0
4      22003   2017               0.0
5      22003   2021               0.0

Statistiques :
count    2404.00
mean        7.43
std        12.08
min         0.00
25%         0.00
50%         0.00
75%        10.80
max        86.00
Name: taux_criminalite, dtype: float64


In [33]:
# ============================================================
# CHARGEMENT NOMBRE D ENTREPRISES
# ============================================================
# On charge le fichier DS_SIDE_STOCKS_ET_COM_2023_data.csv
# qui contient le nombre d entreprises actives par commune.
#
# On affiche les colonnes et quelques lignes pour
# comprendre la structure du fichier et identifier
# les colonnes a utiliser.
# ============================================================

df_ent = pd.read_csv(
    RAW + "DS_SIDE_STOCKS_ET_COM_2023_data.csv",
    sep=";",
    encoding="latin1",
    low_memory=False
)
print("Entreprises charge :", df_ent.shape)
print("Colonnes :", df_ent.columns.tolist())
print()
print("5 premieres lignes :")
print(df_ent.head(5))
print()
print("Valeurs uniques TIME_PERIOD :")
print(sorted(df_ent['TIME_PERIOD'].unique()))

Entreprises charge : (4180500, 7)
Colonnes : ['GEO', 'GEO_OBJECT', 'SIDE_MEASURE', 'ACTIVITY', 'FREQ', 'TIME_PERIOD', 'OBS_VALUE']

5 premieres lignes :
     GEO GEO_OBJECT SIDE_MEASURE ACTIVITY FREQ  TIME_PERIOD  OBS_VALUE
0  01075        COM     UNIT_LOC       GI    A         2017         20
1  90010     BV2022     UNIT_LOC       LZ    A         2015        285
2  01025        COM     UNIT_LOC       _T    A         2016        123
3  91016     BV2022     UNIT_LOC       JZ    A         2014         20
4  01075        COM     UNIT_LOC       KZ    A         2016          4

Valeurs uniques TIME_PERIOD :
[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


In [34]:
# ============================================================
# EXTRACTION ENTREPRISES BRETAGNE
# ============================================================
# Le fichier contient plusieurs types de donnees.
# On filtre uniquement ce dont on a besoin :
#
# GEO_OBJECT = COM : uniquement les communes
#              car le fichier contient aussi des
#              donnees par bassin de vie et autres
#
# ACTIVITY = _T : toutes les activites confondues
#                 car on veut le total d entreprises
#                 et non par secteur d activite
#
# TIME_PERIOD = 2017 et 2021 : nos deux annees
#
# OBS_VALUE : nombre d entreprises actives
#
# On filtre ensuite les communes bretonnes en utilisant
# les 2 premiers caracteres du code commune GEO.
# ============================================================

# Filtrage sur communes, toutes activites, annees 2017 et 2021
df_ent_filtre = df_ent[
    (df_ent['GEO_OBJECT'] == 'COM') &
    (df_ent['ACTIVITY'] == '_T') &
    (df_ent['TIME_PERIOD'].isin([2017, 2021]))
].copy()

print("Apres filtrage :", len(df_ent_filtre), "lignes")

# Conversion code commune en string
df_ent_filtre['GEO'] = df_ent_filtre['GEO'].astype(str).str.zfill(5)

# Extraction code departement
df_ent_filtre['dept'] = df_ent_filtre['GEO'].str[:2]

# Filtrage Bretagne
ent_bretagne = df_ent_filtre[
    df_ent_filtre['dept'].isin(DEPTS_BRETAGNE)
].copy()

print("Apres filtrage Bretagne :", len(ent_bretagne), "lignes")

# Garder uniquement les colonnes utiles
ent_final = ent_bretagne[['GEO', 'TIME_PERIOD', 'OBS_VALUE']].copy()
ent_final.columns = ['code_insee', 'annee', 'nb_entreprises']

print()
print("Verification :")
print(ent_final.head(6))
print()
print("Statistiques :")
print(ent_final['nb_entreprises'].describe().round(2))

Apres filtrage : 69736 lignes
Apres filtrage Bretagne : 2404 lignes

Verification :
       code_insee  annee  nb_entreprises
608952      22009   2021               7
608957      22006   2021              10
609409      22002   2017              35
609904      22030   2021              49
610468      22006   2017               9
611391      22029   2021              31

Statistiques :
count     2404.00
mean       200.91
std        762.58
min          0.00
25%         32.00
50%         70.00
75%        174.25
max      22993.00
Name: nb_entreprises, dtype: float64


In [35]:
# ============================================================
# ASSEMBLAGE DU DATASET FINAL
# ============================================================
# On assemble toutes les sources de donnees en un seul
# dataset final en faisant des jointures sur le code
# commune et l annee.
#
# La strategie est la suivante :
# 1. On part du dataset electoral comme base
# 2. On cree un code_insee complet sur 5 caracteres
# 3. On joint chaque source d indicateurs une par une
#
# how=left : garde toutes les lignes du DataFrame
#            de gauche meme si pas de correspondance
#            dans le DataFrame de droite
# suffixes : quand deux colonnes ont le meme nom
#            pandas ajoute un suffixe pour les distinguer
# ============================================================

# Creer le code_insee complet sur 5 caracteres
electoral['code_insee'] = (
    electoral['code_dept'].astype(str).str.zfill(2) +
    electoral['code_commune'].astype(str).str.zfill(3)
)

print("Dataset electoral :", electoral.shape)
print("Exemple code_insee :", electoral['code_insee'].head(3).tolist())
print()

# Jointure avec la geographie
dataset = electoral.merge(
    geo_bretagne,
    on='code_insee',
    how='left'
)
print("Apres jointure geo :", dataset.shape)

# Preparation population
pop_2017_j = pop_2017[['code_insee', 'population']].copy()
pop_2017_j['annee'] = 2017

pop_2021_j = pop_2021[['code_insee', 'population']].copy()
pop_2021_j['annee'] = 2022

pop_2017_j['code_insee'] = pop_2017_j['code_insee'].astype(str).str.zfill(5)
pop_2021_j['code_insee'] = pop_2021_j['code_insee'].astype(str).str.zfill(5)

pop_all = pd.concat([pop_2017_j, pop_2021_j], axis=0)

# Jointure population
dataset = dataset.merge(
    pop_all,
    on=['code_insee', 'annee'],
    how='left'
)
print("Apres jointure population :", dataset.shape)

# Preparation emploi
emploi_2017_j = emploi_final[['code_insee',
                               'taux_chomage_2017',
                               'pct_diplomes_sup',
                               'pct_sans_diplome']].copy()
emploi_2017_j.columns = ['code_insee', 'taux_chomage',
                          'pct_diplomes_sup', 'pct_sans_diplome']
emploi_2017_j['annee'] = 2017

emploi_2022_j = emploi_final[['code_insee',
                               'taux_chomage_2021',
                               'pct_diplomes_sup',
                               'pct_sans_diplome']].copy()
emploi_2022_j.columns = ['code_insee', 'taux_chomage',
                          'pct_diplomes_sup', 'pct_sans_diplome']
emploi_2022_j['annee'] = 2022

emploi_all = pd.concat([emploi_2017_j, emploi_2022_j], axis=0)

# Jointure emploi
dataset = dataset.merge(
    emploi_all,
    on=['code_insee', 'annee'],
    how='left'
)
print("Apres jointure emploi :", dataset.shape)

# Preparation revenu median
rev_2017_j = rev_2017[['code_insee', 'revenu_median_2017']].copy()
rev_2017_j.columns = ['code_insee', 'revenu_median']
rev_2017_j['code_insee'] = rev_2017_j['code_insee'].astype(str).str.zfill(5)
rev_2017_j['annee'] = 2017

rev_2022_j = rev_2021[['code_insee', 'revenu_median_2021']].copy()
rev_2022_j.columns = ['code_insee', 'revenu_median']
rev_2022_j['code_insee'] = rev_2022_j['code_insee'].astype(str).str.zfill(5)
rev_2022_j['annee'] = 2022

rev_all = pd.concat([rev_2017_j, rev_2022_j], axis=0)

# Jointure revenu
dataset = dataset.merge(
    rev_all,
    on=['code_insee', 'annee'],
    how='left'
)
print("Apres jointure revenu :", dataset.shape)

# Preparation criminalite
crime_2017_j = crime_final[
    crime_final['annee'] == 2017
][['code_insee', 'taux_criminalite']].copy()
crime_2017_j['annee'] = 2017

crime_2022_j = crime_final[
    crime_final['annee'] == 2021
][['code_insee', 'taux_criminalite']].copy()
crime_2022_j['annee'] = 2022

crime_all = pd.concat([crime_2017_j, crime_2022_j], axis=0)

# Jointure criminalite
dataset = dataset.merge(
    crime_all,
    on=['code_insee', 'annee'],
    how='left'
)
print("Apres jointure criminalite :", dataset.shape)

# Preparation entreprises
ent_2017_j = ent_final[
    ent_final['annee'] == 2017
][['code_insee', 'nb_entreprises']].copy()
ent_2017_j['annee'] = 2017

ent_2022_j = ent_final[
    ent_final['annee'] == 2021
][['code_insee', 'nb_entreprises']].copy()
ent_2022_j['annee'] = 2022

ent_all = pd.concat([ent_2017_j, ent_2022_j], axis=0)

# Jointure entreprises
dataset = dataset.merge(
    ent_all,
    on=['code_insee', 'annee'],
    how='left'
)
print("Apres jointure entreprises :", dataset.shape)
print()
print("Colonnes finales :")
print(dataset.columns.tolist())

Dataset electoral : (2440, 14)
Exemple code_insee : ['22001', '22002', '22003']

Apres jointure geo : (2440, 16)
Apres jointure population : (2440, 17)
Apres jointure emploi : (2440, 20)
Apres jointure revenu : (2440, 21)
Apres jointure criminalite : (2440, 22)
Apres jointure entreprises : (2440, 23)

Colonnes finales :
['code_commune', 'nom_commune', 'code_dept', 'annee', 'nb_inscrits', 'nb_votants', 'taux_abstention', 'score_ext_gauche', 'score_gauche', 'score_centre', 'score_droite', 'score_ext_droite', 'gagnant', 'code_insee', 'superficie_km2', 'densite', 'population', 'taux_chomage', 'pct_diplomes_sup', 'pct_sans_diplome', 'revenu_median', 'taux_criminalite', 'nb_entreprises']


In [36]:
# ============================================================
# NETTOYAGE FINAL ET EXPORT CSV
# ============================================================
# On reorganise les colonnes dans l ordre logique
# du dataset final et on gere les valeurs manquantes.
#
# Pour les valeurs manquantes on utilise deux strategies :
# - Pour le revenu median : on remplace par la mediane
#   de la Bretagne car c est une donnee sensible
# - Pour les autres indicateurs : on remplace par la
#   mediane par departement pour garder la coherence
#   geographique
#
# On exporte ensuite le dataset en CSV dans le dossier
# data/processed/ pour l utiliser dans les etapes suivantes
# ============================================================

# Reorganiser les colonnes dans l ordre logique
colonnes_finales = [
    'code_insee',
    'code_commune',
    'nom_commune',
    'code_dept',
    'annee',
    'nb_inscrits',
    'nb_votants',
    'taux_abstention',
    'score_ext_gauche',
    'score_gauche',
    'score_centre',
    'score_droite',
    'score_ext_droite',
    'gagnant',
    'population',
    'superficie_km2',
    'densite',
    'taux_chomage',
    'revenu_median',
    'taux_criminalite',
    'nb_entreprises',
    'pct_diplomes_sup',
    'pct_sans_diplome'
]

dataset_final = dataset[colonnes_finales].copy()

print("Dataset avant nettoyage :", dataset_final.shape)
print()
print("Valeurs manquantes par colonne :")
print(dataset_final.isnull().sum())

Dataset avant nettoyage : (2440, 23)

Valeurs manquantes par colonne :
code_insee           0
code_commune         0
nom_commune          0
code_dept            0
annee                0
nb_inscrits          0
nb_votants           0
taux_abstention      0
score_ext_gauche     0
score_gauche         0
score_centre         0
score_droite         0
score_ext_droite     0
gagnant              0
population          25
superficie_km2      25
densite             25
taux_chomage        28
revenu_median       35
taux_criminalite    36
nb_entreprises      36
pct_diplomes_sup    28
pct_sans_diplome    28
dtype: int64


In [37]:
# ============================================================
# GESTION VALEURS MANQUANTES ET EXPORT
# ============================================================
# On a tres peu de valeurs manquantes ce qui est excellent.
# La plupart concernent les petites communes rurales
# pour lesquelles l INSEE ne publie pas certaines donnees
# pour des raisons de confidentialite statistique.
#
# Strategie de remplacement :
# On remplace les valeurs manquantes par la mediane
# du departement correspondant. La mediane est plus
# robuste que la moyenne car elle n est pas influencee
# par les valeurs extremes des grandes villes.
#
# fillna : remplace les valeurs manquantes
# groupby + transform : calcule la mediane par groupe
#                       et l applique a chaque ligne
#                       du meme groupe
#
# Si apres le remplacement par departement il reste
# des valeurs manquantes on les remplace par la
# mediane generale de la Bretagne.
# ============================================================

# Colonnes a imputer
cols_a_imputer = [
    'population', 'superficie_km2', 'densite',
    'taux_chomage', 'revenu_median', 'taux_criminalite',
    'nb_entreprises', 'pct_diplomes_sup', 'pct_sans_diplome'
]

# Remplacement par la mediane du departement
for col in cols_a_imputer:
    mediane_dept = dataset_final.groupby('code_dept')[col].transform('median')
    dataset_final[col] = dataset_final[col].fillna(mediane_dept)

# Correction superficie = 0 (arrondi fichier source)
# On recalcule depuis population / densite
mask = dataset_final['superficie_km2'] == 0
dataset_final.loc[mask, 'superficie_km2'] = (
    dataset_final.loc[mask, 'population'] / 
    dataset_final.loc[mask, 'densite']
).round(2)
print("Superficies corrigees :", mask.sum())

# Remplacement des valeurs encore manquantes par mediane Bretagne
for col in cols_a_imputer:
    mediane_bretagne = dataset_final[col].median()
    dataset_final[col] = dataset_final[col].fillna(mediane_bretagne)

print("Valeurs manquantes apres nettoyage :")
print(dataset_final.isnull().sum())
print()

# Arrondir les colonnes numeriques a 2 decimales
cols_numeriques = [
    'taux_abstention', 'score_ext_gauche', 'score_gauche',
    'score_centre', 'score_droite', 'score_ext_droite',
    'taux_chomage', 'revenu_median', 'taux_criminalite',
    'pct_diplomes_sup', 'pct_sans_diplome', 'densite'
]
dataset_final[cols_numeriques] = dataset_final[cols_numeriques].round(2)

print("Dimensions finales :", dataset_final.shape)
print()
print("Apercu du dataset final :")
print(dataset_final.head(5))
print()

# Verification doublons
doublons = dataset_final.duplicated(subset=['code_insee', 'annee']).sum()
print("Doublons :", doublons)
assert doublons == 0, "ATTENTION : doublons detectes !"

# Export en CSV
chemin_export = PROCESSED + "dataset_final_bretagne.csv"
dataset_final.to_csv(chemin_export, index=False, encoding='utf-8-sig')

print("Dataset exporte avec succes :")
print(chemin_export)
print()
print("Verification du fichier exporte :")
df_verif = pd.read_csv(chemin_export)
print("Dimensions :", df_verif.shape)
print("Colonnes :", df_verif.columns.tolist())

Superficies corrigees : 4
Valeurs manquantes apres nettoyage :
code_insee          0
code_commune        0
nom_commune         0
code_dept           0
annee               0
nb_inscrits         0
nb_votants          0
taux_abstention     0
score_ext_gauche    0
score_gauche        0
score_centre        0
score_droite        0
score_ext_droite    0
gagnant             0
population          0
superficie_km2      0
densite             0
taux_chomage        0
revenu_median       0
taux_criminalite    0
nb_entreprises      0
pct_diplomes_sup    0
pct_sans_diplome    0
dtype: int64

Dimensions finales : (2440, 23)

Apercu du dataset final :
  code_insee code_commune          nom_commune code_dept  annee nb_inscrits  \
0      22001          001             Allineuc        22   2017         419   
1      22002          002                Andel        22   2017         858   
2      22003          003             Aucaleuc        22   2017         672   
3      22004          004               Bé

In [38]:
# ============================================================
# RESUME ET STATISTIQUES FINALES
# ============================================================
# On affiche un resume complet du dataset final pour
# valider que tout est correct avant de passer a l etape
# suivante qui est le stockage SQL.
# ============================================================
nb_par_commune = dataset_final.groupby('code_insee')['annee'].count()
print("Communes avec 2 annees :", (nb_par_commune == 2).sum())
print("Communes avec 1 annee  :", (nb_par_commune == 1).sum())

print("=" * 60)
print("RESUME ETL - DATASET FINAL BRETAGNE")
print("=" * 60)
print()
print("Dimensions         :", dataset_final.shape)
print("Communes 2017      :", len(dataset_final[dataset_final['annee'] == 2017]))
print("Communes 2022      :", len(dataset_final[dataset_final['annee'] == 2022]))
print("Valeurs manquantes :", dataset_final.isnull().sum().sum())
print()

print("Distribution des gagnants 2017 :")
print(dataset_final[dataset_final['annee'] == 2017]['gagnant'].value_counts())
print()

print("Distribution des gagnants 2022 :")
print(dataset_final[dataset_final['annee'] == 2022]['gagnant'].value_counts())
print()

print("Statistiques descriptives :")
print(dataset_final[[
    'taux_chomage', 'revenu_median',
    'taux_criminalite', 'nb_entreprises',
    'pct_diplomes_sup', 'pct_sans_diplome'
]].describe().round(2))

print()
print("=" * 60)
print("ETL TERMINE - Fichier sauvegarde dans data/processed/")
print("dataset_final_bretagne.csv")
print("=" * 60)

Communes avec 2 annees : 1207
Communes avec 1 annee  : 26
RESUME ETL - DATASET FINAL BRETAGNE

Dimensions         : (2440, 23)
Communes 2017      : 1233
Communes 2022      : 1207
Valeurs manquantes : 0

Distribution des gagnants 2017 :
gagnant
gauche        447
ext_droite    373
centre        351
droite         62
Name: count, dtype: int64

Distribution des gagnants 2022 :
gagnant
ext_droite    586
centre        361
gauche        260
Name: count, dtype: int64

Statistiques descriptives :
       taux_chomage  revenu_median  taux_criminalite  nb_entreprises  \
count       2440.00        2440.00           2440.00         2440.00   
mean           9.88       21430.61              7.33          198.89   
std            3.26        2196.90             12.02          757.11   
min            2.31       13420.00              0.00            0.00   
25%            7.60       19947.50              0.00           32.75   
50%            9.41       21190.00              0.00           69.50   
75%